# Embedding Experiments — Stratified Permutation Tests

Four analyses, all using `stratified_permutation_test` from `orgpackage/tester.py`:

| # | Analysis | Experimental factor | Strata |
|---|----------|--------------------|---------|
| 1 | **Shot count** (similarity only) | 0-shot vs 1-shot vs few-shot | Embedding model |
| 2 | **Model comparison** (similarity) | multilingual-e5 vs qwen vs mistral vs e5-small vs finetuned-me5 | Domain |
| 3 | **Classifier head** (LR vs SVM) | Logistic Regression vs SVM | Embedding model |
| 4 | **Model comparison** (classifiers) | same 5 models | Domain |

Correctness tables are **cached** to `results/correctness_tables/`.  
A single global **Holm correction** is applied across all test results at the end.

## 0. Imports & constants

In [ ]:
import os, sys, ast, warnings
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.metrics.pairwise import cosine_similarity

project_root = os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

from orgpackage.aux import load_experiments, load_trained_model
from orgpackage.config import DOMAIN_CLASSES_CORR
from orgpackage.tester import stratified_permutation_test

warnings.filterwarnings("ignore")

EXPERIMENTS_PATH = "./results/experiments.csv"
EMBEDDINGS_DIR   = "./results/embeddings"
CORRECTNESS_DIR  = "./results/correctness_tables"
DATASET_PATH     = "./data/wikidata_enriched_dataset.csv"

DOMAINS      = ["medical", "administrative", "education"]
ALL_LABELS   = ["hospital", "university_hospital", "local_government",
                "primary_school", "secondary_school"]

os.makedirs(CORRECTNESS_DIR, exist_ok=True)
print("Setup done. Project root:", project_root)

## 1. Load experiments

In [ ]:
exps = load_experiments(EXPERIMENTS_PATH)
emb_exps = exps[exps["Technique"] == "embedding"].copy()

sim_exps = emb_exps[emb_exps["Method"] == "similarity"].copy()
clf_exps = emb_exps[emb_exps["Method"] == "classifier"].copy()

print(f"Similarity experiments : {len(sim_exps)}")
print(f"Classifier experiments : {len(clf_exps)}")

## 2. Load master dataset (ground-truth labels)

> **Bug fix**: embedding CSVs generated on the server may not include class label columns.  
> We always source ground-truth labels from the master dataset and merge by `instance` URI.

In [ ]:
print("Loading master dataset for ground-truth labels…")
master_df = pd.read_csv(DATASET_PATH, low_memory=False)
master_df = master_df.drop_duplicates(subset=["instance"])
master_df = master_df[["instance"] + ALL_LABELS].copy()
for col in ALL_LABELS:
    master_df[col] = master_df[col].fillna(0).astype(int)
print(f"  {len(master_df):,} unique instances loaded.")

## 3. Index experiments by model × n_shot / classifier type

In [ ]:
# ── similarity ──────────────────────────────────────────────────────────────
# {model: {n_shot: [exp_id, ...]}}
sim_index = {}
for _, row in sim_exps.iterrows():
    p = row["Parameters"]
    if not isinstance(p, dict): continue
    sim_index.setdefault(p["model"], {}).setdefault(p["n_shot"], []).append(row["ID"])

# ── classifiers ─────────────────────────────────────────────────────────────
# {model: {clf_type: [exp_id, ...]}   clf_type in {'lr', 'svm'}}
def _clf_type(params):
    if not isinstance(params, dict): return "unknown"
    return "lr" if "solver" in params else ("svm" if "kernel" in params else "unknown")

clf_index = {}
for _, row in clf_exps.iterrows():
    p = row["Parameters"]
    if not isinstance(p, dict): continue
    t = _clf_type(p)
    clf_index.setdefault(p["model"], {}).setdefault(t, []).append(row["ID"])

print("Similarity index:")
for m, shots in sim_index.items():
    print(f"  {m}: {list(shots.keys())}")
print("\nClassifier index:")
for m, types in clf_index.items():
    print(f"  {m}: {list(types.keys())}")

## 4. Helper functions

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Shared: load + parse an embedding CSV, merge with master for labels
# ─────────────────────────────────────────────────────────────────────────────
def _load_embedding_df(model_name, embeddings_dir=EMBEDDINGS_DIR):
    emb_path = os.path.join(embeddings_dir, f"{model_name}_embeddings.csv")
    if not os.path.exists(emb_path):
        raise FileNotFoundError(f"Not found: {emb_path}")
    print(f"    Loading {emb_path}")
    emb_col = f"{model_name}_embedding"
    df = pd.read_csv(emb_path, low_memory=False)

    def _parse(v):
        if v is None or (isinstance(v, float) and np.isnan(v)): return None
        if isinstance(v, str):
            try: return np.array(ast.literal_eval(v))
            except: return None
        return v if isinstance(v, np.ndarray) else None

    df[emb_col] = df[emb_col].apply(_parse)
    valid = df[emb_col].apply(lambda v: v is not None and not np.isnan(v).any())
    df = df[valid][["instance", emb_col]].reset_index(drop=True)

    # Always merge with master dataset → guarantees ALL class label columns
    df = df.merge(master_df, on="instance", how="inner")
    print(f"      → {len(df):,} instances with valid embeddings after master merge.")
    return df, emb_col


# ─────────────────────────────────────────────────────────────────────────────
# SIMILARITY: per-instance correctness using prototype + cosine similarity
# ─────────────────────────────────────────────────────────────────────────────
def _classify_similarity(x_emb, prototypes, threshold, structure, classes):
    is_multiclass = (structure == "2-multiclass")
    x = np.array(x_emb).reshape(1, -1)
    sims = {}
    for cls in classes:
        proto = prototypes.get(cls)
        best = -1.0
        if proto is None:
            sims[cls] = -1.0; continue
        if isinstance(proto, dict):
            for _, p in proto.items():
                if p is None: continue
                if isinstance(p, list): p = np.array(p)
                if not isinstance(p, np.ndarray) or np.isnan(p).any(): continue
                try: best = max(best, cosine_similarity(x, p.reshape(1,-1))[0][0])
                except: pass
        else:
            if isinstance(proto, list): proto = np.array(proto)
            if isinstance(proto, np.ndarray) and not np.isnan(proto).any():
                try: best = cosine_similarity(x, proto.reshape(1,-1))[0][0]
                except: pass
        sims[cls] = best

    preds = {cls: 0 for cls in classes}
    if all(v == -1.0 for v in sims.values()): return preds
    if is_multiclass:
        for cls in classes:
            if sims.get(cls, -1.0) >= threshold: preds[cls] = 1
    else:
        valid = {k: v for k, v in sims.items() if v != -1.0}
        if valid:
            best_cls = max(valid, key=valid.get)
            if valid[best_cls] >= threshold: preds[best_cls] = 1
    return preds


def build_similarity_correctness_table(model_name, exp_ids, exps_df,
                                         embeddings_dir=EMBEDDINGS_DIR):
    emb_df, emb_col = _load_embedding_df(model_name, embeddings_dir)
    cols = {}
    for exp_id in exp_ids:
        row = exps_df[exps_df["ID"] == exp_id]
        if row.empty: print(f"    WARNING: {exp_id} not found."); continue
        exp = row.iloc[0]
        classes   = DOMAIN_CLASSES_CORR[exp["Domain"]]
        params    = exp["Parameters"]
        threshold = 1.0 - params["distance"]
        structure = params["structure"]
        prototypes= params["prototypes"]

        hit = []
        for _, r in emb_df.iterrows():
            y_true = {cls: int(r[cls]) for cls in classes}
            preds  = _classify_similarity(r[emb_col], prototypes, threshold, structure, classes)
            hit.append((r["instance"], int(all(preds[c] == y_true[c] for c in classes))))

        s = pd.Series([h[1] for h in hit], index=[h[0] for h in hit],
                      name=exp_id, dtype=int)
        cols[exp_id] = s
        print(f"    {exp_id} ({exp['Domain']:15s}): {len(s):,} rows, "
              f"mean={s.mean():.3f}")

    if not cols: raise ValueError(f"No predictions for model '{model_name}'.")
    return pd.DataFrame(cols).fillna(0).astype(int)


# ─────────────────────────────────────────────────────────────────────────────
# CLASSIFIER: per-instance correctness using a trained sklearn model
# ─────────────────────────────────────────────────────────────────────────────
def _predict_classifier(clf_params, X, classes, structure):
    """Run predictions (handles nested-class for medical)."""
    n = len(X)
    y_pred = np.zeros((n, len(classes)), dtype=int)

    if structure == "nested-class":
        hosp_clf = load_trained_model(clf_params["trained_classifier_hospital"])
        univ_clf = load_trained_model(clf_params["trained_classifier_university_hospital"])
        hi = classes.index("hospital")
        ui = classes.index("university_hospital")
        y_h = hosp_clf.predict(X)
        y_pred[:, hi] = y_h
        pos = np.where(y_h == 1)[0]
        if len(pos):
            y_pred[pos, ui] = univ_clf.predict(X[pos])
    else:
        clf = load_trained_model(clf_params["trained_classifier"])
        raw = clf.predict(X)
        if raw.ndim == 1 and len(classes) == 1:
            y_pred[:, 0] = raw
        elif raw.ndim == 1:
            # multiclass single output → one-hot against index
            for i, r in enumerate(raw):
                if r < len(classes): y_pred[i, r] = 1
        else:
            y_pred = raw.astype(int)
    return y_pred


def _clf_files_missing(params, structure):
    """Return True if the required trained classifier pickle files are missing."""
    if structure == "nested-class":
        for key in ["trained_classifier_hospital", "trained_classifier_university_hospital"]:
            path = params.get(key)
            if not path or not os.path.exists(str(path)): return True
    else:
        path = params.get("trained_classifier")
        if not path or not os.path.exists(str(path)): return True
    return False


def build_classifier_correctness_table(model_name, exp_ids, exps_df,
                                         embeddings_dir=EMBEDDINGS_DIR):
    emb_df, emb_col = _load_embedding_df(model_name, embeddings_dir)

    # Build X once (all instances, ordered)
    X    = np.stack(emb_df[emb_col].values)
    instances = emb_df["instance"].values

    cols = {}
    for exp_id in exp_ids:
        row = exps_df[exps_df["ID"] == exp_id]
        if row.empty: print(f"    WARNING: {exp_id} not found."); continue
        exp       = row.iloc[0]
        classes   = DOMAIN_CLASSES_CORR[exp["Domain"]]
        params    = exp["Parameters"]
        structure = params.get("structure", "2-class")

        if _clf_files_missing(params, structure):
            print(f"    SKIP {exp_id}: classifier pickle not found (server-only).")
            continue

        try:
            y_pred = _predict_classifier(params, X, classes, structure)
        except Exception as e:
            print(f"    ERROR {exp_id}: {e}"); continue

        y_true  = emb_df[classes].values
        correct = (y_pred == y_true).all(axis=1).astype(int)
        s = pd.Series(correct, index=instances, name=exp_id, dtype=int)
        cols[exp_id] = s
        print(f"    {exp_id} ({exp['Domain']:15s}): {len(s):,} rows, mean={s.mean():.3f}")

    if not cols: raise ValueError(f"No classifier predictions for model '{model_name}'.")
    return pd.DataFrame(cols).fillna(0).astype(int)


# ─────────────────────────────────────────────────────────────────────────────
# Lookup helpers
# ─────────────────────────────────────────────────────────────────────────────
def _get_id(index, model, key, domain):
    """Return experiment ID matching (model, key, domain-prefix)."""
    prefix = domain[:3]
    for eid in index.get(model, {}).get(key, []):
        if eid.startswith(prefix): return eid
    return None


print("Helpers defined.")

## 5. Build & cache similarity correctness tables

In [ ]:
FORCE_RECOMPUTE_SIM = False

sim_ct = {}   # {model: correctness_df}

for model, shot_groups in sim_index.items():
    cache = os.path.join(CORRECTNESS_DIR, f"{model}_sim_correctness.csv")

    if not FORCE_RECOMPUTE_SIM and os.path.exists(cache):
        print(f"[CACHE] {model} similarity: {cache}")
        ct = pd.read_csv(cache, index_col=0).astype(int)
        sim_ct[model] = ct
        print(f"  → shape {ct.shape}")
        continue

    emb_path = os.path.join(EMBEDDINGS_DIR, f"{model}_embeddings.csv")
    if not os.path.exists(emb_path):
        print(f"[SKIP] {model}: embeddings file not found.")
        continue

    all_ids = [eid for ids in shot_groups.values() for eid in ids]
    print(f"\n[COMPUTE] {model} similarity: {all_ids}")
    try:
        ct = build_similarity_correctness_table(model, all_ids, sim_exps)
        sim_ct[model] = ct
        ct.to_csv(cache)
        print(f"  → saved {cache}  shape={ct.shape}")
    except Exception as e:
        print(f"  ERROR: {e}")

print("\nSimilarity models available:", list(sim_ct.keys()))

## 6. Build & cache classifier correctness tables

In [ ]:
FORCE_RECOMPUTE_CLF = False

clf_ct = {}   # {model: correctness_df}

for model, type_groups in clf_index.items():
    cache = os.path.join(CORRECTNESS_DIR, f"{model}_clf_correctness.csv")

    if not FORCE_RECOMPUTE_CLF and os.path.exists(cache):
        print(f"[CACHE] {model} classifier: {cache}")
        ct = pd.read_csv(cache, index_col=0).astype(int)
        clf_ct[model] = ct
        print(f"  → shape {ct.shape}")
        continue

    emb_path = os.path.join(EMBEDDINGS_DIR, f"{model}_embeddings.csv")
    if not os.path.exists(emb_path):
        print(f"[SKIP] {model}: embeddings file not found.")
        continue

    all_ids = [eid for ids in type_groups.values() for eid in ids]
    print(f"\n[COMPUTE] {model} classifier: {all_ids}")
    try:
        ct = build_classifier_correctness_table(model, all_ids, clf_exps)
        clf_ct[model] = ct
        ct.to_csv(cache)
        print(f"  → saved {cache}  shape={ct.shape}")
    except Exception as e:
        print(f"  ERROR: {e}")

print("\nClassifier models available:", list(clf_ct.keys()))

---
## Analysis 1 — Shot count comparison (similarity)

**Factor**: `n_shot` ∈ {`0_shot`, `1_shot`, `few_shot`}  
**Strata**: one per embedding model  
**Tests**: three pairwise comparisons per domain + pooled

In [ ]:
SHOT_PAIRS = [
    ("1_shot",   "0_shot",   "1-shot vs 0-shot"),
    ("few_shot", "1_shot",   "few-shot vs 1-shot"),
    ("few_shot", "0_shot",   "few-shot vs 0-shot"),
]


def run_shot_comparison(shot_a, shot_b, label, n_perm=10_000):
    results = []
    for domain in DOMAINS:
        strata = []
        for model, ct in sim_ct.items():
            id_a = _get_id(sim_index, model, shot_a, domain)
            id_b = _get_id(sim_index, model, shot_b, domain)
            if not id_a or not id_b: continue
            if id_a not in ct.columns or id_b not in ct.columns: continue
            strata.append(ct[[id_a, id_b]].rename(columns={id_a: "A", id_b: "B"}))
        if not strata: continue
        obs, p = stratified_permutation_test(strata, "A", "B",
                                              n_perm=n_perm, statistic="pooled")
        a_all = np.concatenate([s["A"].values for s in strata])
        b_all = np.concatenate([s["B"].values for s in strata])
        results.append({"analysis": "A1_shot", "comparison": label,
                         "scope": domain, "n_strata": len(strata),
                         "n_instances": sum(len(s) for s in strata),
                         "mean_a": round(a_all.mean(), 4),
                         "mean_b": round(b_all.mean(), 4),
                         "obs_diff": round(obs, 4), "p_value": p})

    # pooled across all domains
    pooled = []
    for model, ct in sim_ct.items():
        subs = []
        for domain in DOMAINS:
            id_a = _get_id(sim_index, model, shot_a, domain)
            id_b = _get_id(sim_index, model, shot_b, domain)
            if id_a and id_b and id_a in ct.columns and id_b in ct.columns:
                subs.append(ct[[id_a, id_b]].rename(columns={id_a: "A", id_b: "B"}))
        if subs: pooled.append(pd.concat(subs))
    if pooled:
        obs, p = stratified_permutation_test(pooled, "A", "B",
                                              n_perm=n_perm, statistic="pooled")
        a_all = np.concatenate([s["A"].values for s in pooled])
        b_all = np.concatenate([s["B"].values for s in pooled])
        results.append({"analysis": "A1_shot", "comparison": label,
                         "scope": "all domains", "n_strata": len(pooled),
                         "n_instances": sum(len(s) for s in pooled),
                         "mean_a": round(a_all.mean(), 4),
                         "mean_b": round(b_all.mean(), 4),
                         "obs_diff": round(obs, 4), "p_value": p})
    return pd.DataFrame(results)


N_PERM = 10_000
a1_results = pd.concat(
    [run_shot_comparison(a, b, lbl, N_PERM) for a, b, lbl in SHOT_PAIRS],
    ignore_index=True
)
print("Analysis 1 done.")
display(a1_results[["comparison", "scope", "n_strata", "n_instances",
                     "mean_a", "mean_b", "obs_diff", "p_value"]])

---
## Analysis 2 — Model comparison (similarity)

**Factor**: embedding model  
**Strata**: one per domain (3 strata)  
**Repeated for each n_shot level**

In [ ]:
def build_model_strata_similarity(n_shot):
    """
    For a fixed n_shot, return a dict {domain: df}
    where df has one column per model (model name as column name).
    Only instances present in ALL models for that domain are kept (inner join).
    """
    domain_strata = {}
    for domain in DOMAINS:
        frames = {}
        for model, ct in sim_ct.items():
            exp_id = _get_id(sim_index, model, n_shot, domain)
            if exp_id and exp_id in ct.columns:
                frames[model] = ct[[exp_id]].rename(columns={exp_id: model})
        if len(frames) < 2:
            print(f"  Skipping {domain} / {n_shot}: only {len(frames)} model(s) available.")
            continue
        # Inner join across models
        combined = list(frames.values())[0]
        for f in list(frames.values())[1:]:
            combined = combined.join(f, how="inner")
        domain_strata[domain] = combined
        print(f"  {domain} / {n_shot}: {len(combined):,} instances × {list(combined.columns)} models")
    return domain_strata


def run_model_pairwise(domain_strata, analysis_tag, n_perm=10_000):
    """All-pairwise model comparisons, strata = domains."""
    if not domain_strata: return pd.DataFrame()
    strata_list = list(domain_strata.values())
    common_models = sorted(set.intersection(*(set(s.columns) for s in strata_list)))
    results = []
    for model_a, model_b in combinations(common_models, 2):
        obs, p = stratified_permutation_test(
            [s[[model_a, model_b]].rename(columns={model_a: "A", model_b: "B"})
             for s in strata_list],
            "A", "B", n_perm=n_perm, statistic="pooled"
        )
        a_all = np.concatenate([s[model_a].values for s in strata_list])
        b_all = np.concatenate([s[model_b].values for s in strata_list])
        results.append({"analysis": analysis_tag,
                         "model_a": model_a, "model_b": model_b,
                         "n_strata": len(strata_list),
                         "n_instances": sum(len(s) for s in strata_list),
                         "mean_a": round(a_all.mean(), 4),
                         "mean_b": round(b_all.mean(), 4),
                         "obs_diff": round(obs, 4), "p_value": p})
    return pd.DataFrame(results)


SHOT_LEVELS = ["0_shot", "1_shot", "few_shot"]
a2_parts = []
for n_shot in SHOT_LEVELS:
    print(f"\n── {n_shot} ──")
    strata = build_model_strata_similarity(n_shot)
    tag = f"A2_model_sim_{n_shot}"
    df = run_model_pairwise(strata, tag, N_PERM)
    if not df.empty:
        df["n_shot"] = n_shot
        a2_parts.append(df)

a2_results = pd.concat(a2_parts, ignore_index=True) if a2_parts else pd.DataFrame()
print("\nAnalysis 2 done.")
display(a2_results)

---
## Analysis 3 — Classifier head comparison (LR vs SVM)

**Factor**: classifier type (`lr` vs `svm`)  
**Strata**: one per model (controlling for embedding backbone)  
Within each model-stratum, all three domains are pooled.

In [ ]:
def build_lr_svm_strata():
    """
    Return a list of DataFrames, one per model that has BOTH lr and svm.
    Each df has columns ['lr', 'svm'], rows = all instances across all 3 domains
    where both classifier types are available.
    """
    strata = []
    for model, ct in clf_ct.items():
        lr_cols, svm_cols = [], []
        for domain in DOMAINS:
            lr_id  = _get_id(clf_index, model, "lr",  domain)
            svm_id = _get_id(clf_index, model, "svm", domain)
            if lr_id and lr_id in ct.columns:   lr_cols.append(lr_id)
            if svm_id and svm_id in ct.columns: svm_cols.append(svm_id)

        if not lr_cols or not svm_cols:
            print(f"  SKIP {model}: missing lr ({lr_cols}) or svm ({svm_cols}) experiments.")
            continue

        # Average correctness across all available domains for each head type
        lr_mean  = ct[lr_cols].mean(axis=1).rename("lr")
        svm_mean = ct[svm_cols].mean(axis=1).rename("svm")
        sub = pd.concat([lr_mean, svm_mean], axis=1).dropna()
        strata.append(sub)
        print(f"  {model}: {len(sub):,} instances  "
              f"lr_cols={lr_cols}  svm_cols={svm_cols}")
    return strata


print("Building LR vs SVM strata…")
lr_svm_strata = build_lr_svm_strata()

if lr_svm_strata:
    obs, p = stratified_permutation_test(lr_svm_strata, "lr", "svm",
                                          n_perm=N_PERM, statistic="pooled")
    lr_all  = np.concatenate([s["lr"].values  for s in lr_svm_strata])
    svm_all = np.concatenate([s["svm"].values for s in lr_svm_strata])
    a3_results = pd.DataFrame([{
        "analysis": "A3_clf_head",
        "comparison": "LR vs SVM",
        "scope": "all domains (pooled)",
        "n_strata": len(lr_svm_strata),
        "n_instances": sum(len(s) for s in lr_svm_strata),
        "mean_lr":  round(lr_all.mean(), 4),
        "mean_svm": round(svm_all.mean(), 4),
        "obs_diff": round(obs, 4),
        "p_value": p,
    }])
    print("\nAnalysis 3 done.")
    display(a3_results)
else:
    a3_results = pd.DataFrame()
    print("Analysis 3: no data (classifier pickle files not available locally).")

---
## Analysis 4 — Model comparison (classifiers)

**Factor**: embedding model  
**Strata**: one per domain  
**Repeated separately for LR experiments and SVM experiments**

In [ ]:
def build_model_strata_classifier(clf_type):
    """
    For a fixed classifier type ('lr' or 'svm'),
    return {domain: df} where df has one column per model.
    """
    domain_strata = {}
    for domain in DOMAINS:
        frames = {}
        for model, ct in clf_ct.items():
            exp_id = _get_id(clf_index, model, clf_type, domain)
            if exp_id and exp_id in ct.columns:
                frames[model] = ct[[exp_id]].rename(columns={exp_id: model})
        if len(frames) < 2:
            print(f"  Skipping {domain} / {clf_type}: only {len(frames)} model(s) available.")
            continue
        combined = list(frames.values())[0]
        for f in list(frames.values())[1:]:
            combined = combined.join(f, how="inner")
        domain_strata[domain] = combined
        print(f"  {domain} / {clf_type}: {len(combined):,} instances × {list(combined.columns)}")
    return domain_strata


a4_parts = []
for clf_type in ["lr", "svm"]:
    print(f"\n── classifier type: {clf_type.upper()} ──")
    strata = build_model_strata_classifier(clf_type)
    tag = f"A4_model_clf_{clf_type}"
    df = run_model_pairwise(strata, tag, N_PERM)
    if not df.empty:
        df["clf_type"] = clf_type
        a4_parts.append(df)

a4_results = pd.concat(a4_parts, ignore_index=True) if a4_parts else pd.DataFrame()
print("\nAnalysis 4 done.")
if not a4_results.empty:
    display(a4_results)
else:
    print("  No data (classifier pickles not available locally).")

---
## Global Holm correction across all analyses

In [ ]:
from statsmodels.stats.multitest import multipletests

# Collect all p-values (rename columns to a common schema)
def _standardise(df, analysis):
    if df.empty: return pd.DataFrame()
    d = df.copy()
    d["analysis"] = analysis
    # ensure a 'comparison' column exists
    if "comparison" not in d.columns:
        if "model_a" in d.columns:
            d["comparison"] = d["model_a"] + " vs " + d["model_b"]
    if "scope" not in d.columns:
        d["scope"] = "all"
    return d[["analysis", "comparison", "scope",
               "n_strata", "n_instances",
               "obs_diff", "p_value"]]

all_parts = [
    _standardise(a1_results.rename(columns={"comparison": "comparison"}), "A1_shot"),
    _standardise(a2_results, "A2_model_sim"),
    _standardise(a3_results, "A3_clf_head"),
    _standardise(a4_results, "A4_model_clf"),
]
all_results = pd.concat([p for p in all_parts if not p.empty], ignore_index=True)

if not all_results.empty:
    reject, p_corr, _, _ = multipletests(all_results["p_value"], method="holm")
    all_results["p_corrected"] = p_corr
    all_results["significant"] = reject

    pd.set_option("display.max_colwidth", 35)
    pd.set_option("display.float_format", "{:.4f}".format)
    print(f"Total tests: {len(all_results)}  |  Significant (Holm p<0.05): {reject.sum()}")
    display(all_results)
else:
    print("No results to correct.")

## Save all results

In [ ]:
os.makedirs("results", exist_ok=True)
out = "results/embedding_permutation_tests.csv"
all_results.to_csv(out, index=False)
print(f"Saved {len(all_results)} rows → {out}")

## Visualisation — one panel per analysis

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

if all_results.empty:
    print("Nothing to plot.")
else:
    analyses = all_results["analysis"].unique()
    fig, axes = plt.subplots(1, len(analyses),
                              figsize=(5 * len(analyses), 5),
                              sharey=False)
    if len(analyses) == 1: axes = [axes]

    for ax, ana in zip(axes, analyses):
        sub = all_results[all_results["analysis"] == ana].copy()
        labels = sub["comparison"].tolist()
        colors = ["#d62728" if s else "#aec7e8" for s in sub["significant"]]
        bars = ax.bar(range(len(sub)), sub["obs_diff"],
                      color=colors, edgecolor="k", linewidth=0.5)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_xticks(range(len(sub)))
        ax.set_xticklabels([l.replace(" vs ", "\nvs ") for l in labels],
                           fontsize=7, rotation=30, ha="right")
        ax.set_title(ana.replace("_", " "), fontsize=9, fontweight="bold")
        ax.set_ylabel("obs diff (A − B)")
        for bar, (_, row) in zip(bars, sub.iterrows()):
            p_str = "p<0.001" if row["p_corrected"] < 0.001 else f"p={row['p_corrected']:.3f}"
            offset = 0.002 if row["obs_diff"] >= 0 else -0.012
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + offset,
                    p_str, ha="center", va="bottom", fontsize=6.5)

    sig_p   = mpatches.Patch(color="#d62728", label="Significant (Holm p<0.05)")
    insig_p = mpatches.Patch(color="#aec7e8", label="Not significant")
    fig.legend(handles=[sig_p, insig_p], loc="lower center",
               ncol=2, fontsize=8, bbox_to_anchor=(0.5, -0.08))
    fig.suptitle("Embedding Experiments — Stratified Permutation Tests (Holm)",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    os.makedirs("figures", exist_ok=True)
    fig.savefig("figures/embedding_permutation_tests.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → figures/embedding_permutation_tests.png")

## Diagnostic — per-model / per-domain breakdown (unadjusted)

In [ ]:
diag_rows = []
rng = np.random.default_rng(42)

# A1: shot-count pairs per model × domain
for shot_a, shot_b, label in SHOT_PAIRS:
    for model, ct in sim_ct.items():
        for domain in DOMAINS:
            id_a = _get_id(sim_index, model, shot_a, domain)
            id_b = _get_id(sim_index, model, shot_b, domain)
            if not id_a or not id_b or id_a not in ct.columns or id_b not in ct.columns: continue
            a, b = ct[id_a].values.astype(float), ct[id_b].values.astype(float)
            diff = a - b; obs = diff.mean()
            signs = rng.choice([-1, 1], size=(N_PERM, len(diff)))
            p = max((np.abs((signs*diff).mean(axis=1)) >= abs(obs)).mean(), 1/N_PERM)
            diag_rows.append({"analysis": "A1", "comparison": label,
                               "model": model, "domain": domain,
                               "mean_a": round(a.mean(),4), "mean_b": round(b.mean(),4),
                               "obs_diff": round(obs,4), "p_unadj": p,
                               "id_a": id_a, "id_b": id_b})

diag_df = pd.DataFrame(diag_rows)
print("Diagnostic (A1 shot-count, unadjusted p-values):")
display(diag_df)